In [ ]:
import pandas as pd
import anndata as ad
from anndata import read_zarr
import scanpy as sc

Reading File, and adding annotations. For some reason, the gene id names have a tendency to disapear, so i just rerun it when needed just in case.

In [ ]:
adata = read_zarr("../integration/Integrated.zarr")

In [ ]:
annot = sc.queries.biomart_annotations(
    "hsapiens",
    ["ensembl_gene_id", "external_gene_name", "description"],
    host="www.ensembl.org"
)
annot = annot.drop_duplicates(subset=['external_gene_name'], keep='first')
ensembl_to_symbol = annot.set_index("ensembl_gene_id")["external_gene_name"]
ensembl_to_description = annot.set_index("ensembl_gene_id")["description"]

In [ ]:
adata.var['geneID'] = adata.var.index.map(ensembl_to_symbol)
adata.var['description'] = adata.var.index.map(ensembl_to_description)
adata.var['ensembl'] = adata.var_names
adata.var_names = adata.var['geneID']

Running rank genes groups. This compares gene expression of clusters in the "groups" list compared to the "reference" cluster. In this case, we are comparing early iPSC-CMs, ventricular, and atrial cardiomyocytes, to the mature iPSC-CMs. Then we are grabbing the outputs of these, filtering for high log fold changes, and pvalues. The pvalues aren't especially relevant here, mainly because there are so many cells in these groups, the p value becomes astronomically small.

In [ ]:
sc.tl.rank_genes_groups(
    adata, 
    groupby='leiden', 
    groups=['1', '3', '5'], 
    reference='0', 
    method='wilcoxon',
    use_raw=False,
    layer = "SCVI_normalized"
)

In [ ]:
matureiPSC_VentCM = sc.get.rank_genes_groups_df(adata, group="3")
matureiPSC_VentCM = matureiPSC_VentCM[
    (abs(matureiPSC_VentCM['logfoldchanges']) > 1.5) & 
    (matureiPSC_VentCM['pvals_adj'] < 1e-5)
]
matureiPSC_VentCM["abslfc"] = abs(matureiPSC_VentCM["logfoldchanges"]).copy()
matureiPSC_VentCM = matureiPSC_VentCM.sort_values(by='abslfc', ascending=False)

In [ ]:
matureiPSC_VentCM.to_csv("VentCMs.csv")

In [ ]:
matureiPSC_ATRCM = sc.get.rank_genes_groups_df(adata, group="5")
matureiPSC_ATRCM = matureiPSC_ATRCM[
    (abs(matureiPSC_ATRCM['logfoldchanges']) > 1.5) & 
    (matureiPSC_ATRCM['pvals_adj'] < 1e-5)
]
matureiPSC_ATRCM["abslfc"] = abs(matureiPSC_ATRCM["logfoldchanges"]).copy()
matureiPSC_ATRCM = matureiPSC_ATRCM.sort_values(by='abslfc', ascending=False)

In [ ]:
matureiPSC_ATRCM.to_csv("ARTCMs.csv")

In [ ]:
matureiPSC_immatureiPSC = sc.get.rank_genes_groups_df(adata, group="1")
matureiPSC_immatureiPSC = matureiPSC_immatureiPSC[
    (abs(matureiPSC_immatureiPSC['logfoldchanges']) > 1.5) & 
    (matureiPSC_immatureiPSC['pvals_adj'] < 1e-5)
]
matureiPSC_immatureiPSC["abslfc"] = abs(matureiPSC_immatureiPSC["logfoldchanges"]).copy()
earlyCM = matureiPSC_immatureiPSC.sort_values(by='abslfc', ascending=False)

In [ ]:

matureiPSC_immatureiPSC.to_csv("earlyCMs.csv")

merging all of the lists. We are only keeping genes that are:
1. either both upregulated, or both downregulated in ventricular and atrial cardiomyocytes compared to mature iPSC-CMs
2. On the opposite side of upregulation/downregulation in early iPSC-CMs compared to mature iPSC-CMs than the atrial/ventricular CMs.

In [ ]:
earlyToMat = matureiPSC_VentCM.merge(matureiPSC_ATRCM, on= 'names')
earlyToMat = earlyToMat[~ earlyToMat['names'].str.startswith('LINC')]

In [ ]:
earlyToMat = earlyToMat[earlyToMat['logfoldchanges_y'] * earlyToMat['logfoldchanges_x'] > 0].copy()

In [ ]:
earlyToMat = earlyToMat.merge(matureiPSC_immatureiPSC, on= 'names')

In [ ]:
earlyToMat = earlyToMat[earlyToMat['logfoldchanges_x'] * earlyToMat['logfoldchanges'] < 0].copy()

Making a new DataFrame and grabbing/renaming all the variables for ease of reading, and adding descriptions

In [ ]:
refinedAnalysis = pd.DataFrame()
refinedAnalysis['GeneID'] = earlyToMat['names']
refinedAnalysis['logfc_Native_CM_Ventricular'] = earlyToMat['logfoldchanges_x']
refinedAnalysis['logfc_Native_CM_Atrial'] = earlyToMat['logfoldchanges_y']
refinedAnalysis['logfc_Immature_CM'] = earlyToMat['logfoldchanges']
refinedAnalysis['description'] = refinedAnalysis['GeneID'].map(annot.set_index('external_gene_name')['description'])

In [ ]:
refinedAnalysis.to_csv("DGELIST.csv")

Seperating upregulated genes and downregulated genes to make a "score" metric of maturity/immaturity to map on a umap.

In [ ]:
upregDEG = refinedAnalysis[refinedAnalysis.logfc_Native_CM_Ventricular > 0].GeneID.tolist()
downregDEG = refinedAnalysis[refinedAnalysis.logfc_Native_CM_Ventricular < 0].GeneID.tolist()

In [ ]:
sc.tl.score_genes(
    adata,
    gene_list=upregDEG,
    score_name="maturationScore",
    use_raw=False,
    layer="SCVI_normalized",
)

In [ ]:
sc.tl.score_genes(
    adata,
    gene_list=downregDEG,
    score_name="immaturationScore",
    use_raw=False,
    layer="SCVI_normalized",
)

In [ ]:
sc.tl.score_genes(
    adata,
    gene_list=downregDEG,
    score_name="immaturationScore",
    use_raw=False,
    layer="SCVI_normalized",
)

In [ ]:
sc.pl.umap(
    adata,
    color=["maturationScore"],
    ncols=2,
    frameon=False,
    cmap="magma",
    vmax="p82",
    save="maturation_metrics.png",
)

In [ ]:
sc.pl.umap(
    adata,
    color=["immaturationScore"],
    ncols=2,
    frameon=False,
    cmap="magma",
    vmax="p82",
    save="immaturation_metrics.png",
)